In [1]:
pip install -q -U langchain-community langchain-huggingface langchain-text-splitters pypdf faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import torch
import csv
import faiss
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

PDF_PATH = '' # insert location of folder with pdfs
CSV_INPUT = "" # insert location of dataset_clean.csv
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

print("Loading laws ...")
loader = DirectoryLoader(
    PDF_PATH, 
    glob="./*.pdf", 
    loader_cls=PyPDFLoader,
    show_progress=True,
    use_multithreading=True
)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
texts = text_splitter.split_documents(docs)

model_kwargs = {'device': 'cuda'} if torch.cuda.is_available() else {}
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs=model_kwargs
)

vector_db = FAISS.from_documents(texts, embeddings)
print("Database ready ...")

print("Loading Mistral model (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

df = pd.read_csv(CSV_INPUT)
results = []

print(f"Starting inference...")

for i, row in df.iterrows():
    user_query = str(row['prompt'])
    
    # Search für top 2 paragraphs
    search_results = vector_db.similarity_search(user_query, k=2)
    context = "\n".join([doc.page_content for doc in search_results])
    
    rag_prompt = f"<s>[INST] Du bist ein Experte für österreichisches Steuerrecht. Nutze ausschließlich die folgenden Information, um die Frage präzise zu beantworten:\n\n{context}\n\nFrage: {user_query} [/INST] Antwort:"
    inputs = tokenizer(rag_prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            min_new_tokens=20, 
            temperature=0.2,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # extract answer
    if "Antwort:" in full_text:
        ans = full_text.split("Antwort:")[-1].strip()
        ans = ans.split("Frage:")[0].split("[INST]")[0].strip()
        ans = ans.replace("\n", " ")
    else:
        ans = full_text.replace("\n", " ")
        
    results.append(ans)

    del inputs, outputs
    torch.cuda.empty_cache()
    
    if (i + 1) % 10 == 0:
        print(f"Completed: {i+1}/{len(df)}")

# save resuluts
df['answer'] = results
df[['id', 'answer']].to_csv("results_with_rag_final.csv", index=False, encoding='utf-8-sig')
print("Completed. File saved as results_with_rag_final.csv")

Loading laws ...


100%|██████████| 11/11 [05:14<00:00, 28.56s/it]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Database ready ...
Loading Mistral model (4-bit)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Starting inference...
Completed: 10/643
Completed: 20/643
Completed: 30/643
Completed: 40/643
Completed: 50/643
Completed: 60/643
Completed: 70/643
Completed: 80/643
Completed: 90/643
Completed: 100/643
Completed: 110/643
Completed: 120/643
Completed: 130/643
Completed: 140/643
Completed: 150/643
Completed: 160/643
Completed: 170/643
Completed: 180/643
Completed: 190/643
Completed: 200/643
Completed: 210/643
Completed: 220/643
Completed: 230/643
Completed: 240/643
Completed: 250/643
Completed: 260/643
Completed: 270/643
Completed: 280/643
Completed: 290/643
Completed: 300/643
Completed: 310/643
Completed: 320/643
Completed: 330/643
Completed: 340/643
Completed: 350/643
Completed: 360/643
Completed: 370/643
Completed: 380/643
Completed: 390/643
Completed: 400/643
Completed: 410/643
Completed: 420/643
Completed: 430/643
Completed: 440/643
Completed: 450/643
Completed: 460/643
Completed: 470/643
Completed: 480/643
Completed: 490/643
Completed: 500/643
Completed: 510/643
Completed: 520/643